## Convert the OpenAI Prompts to DSPY

This notebook is used for testing and iteration to figure out what modules and signatures I should create to translate the openAI prompts into DSPy so I don't have to work directly with both DSPy and the OpenAI API

### Setting up DSPy

In [ ]:
# Import libraries
import dspy
import json
import random
import tiktoken

import os
from dotenv import load_dotenv
load_dotenv()
open_ai_api_key = os.getenv("OPENAI_API_KEY")


c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [ ]:
# # create gpt-4 model
# gpt4 = dspy.OpenAI(model='gpt-4-0125-preview', max_tokens=4000, api_key=open_ai_api_key)  
# dspy.configure(lm=gpt4)

# gpt4("which openai model are you? Are you gpt4?")

["I am based on OpenAI's technology, and my responses are generated based on the capabilities of models up to GPT-4. However, I don't have real-time updates or the ability to confirm the specific version I am at any given moment. My design is to provide information and answer questions to the best of my training and capabilities, which are reflective of the advancements up to and including GPT-4."]

In [ ]:
# Set up the LM (https://dspy-docs.vercel.app/api/language_model_clients/OpenAI)
gpt3_turbo = dspy.OpenAI(model='gpt-3.5-turbo', max_tokens=4000, api_key=open_ai_api_key)  
dspy.configure(lm=gpt3_turbo)

### Declaring some global constants

In [4]:
constants = {
    "question_json_format": """{
        "cell_type": "question",
        "response_format": "open" or "closed",
        "main_text": string,
        "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
    }""",
    "section_json_format": """{
        "id": section number starting from 0,
        "title": string,
        "time_estimate": number of minutes,
        "cells": [
            {
                "cell_type": "question" or "text",
                "response_format": "open" or "closed",
                "main_text": string,
                "rationale": string,
                "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
            }
        ]
    }""",
    "json_formatting_message": """Do not include any markdown formatting like triple quotes or backticks around your JSON output. If your output includes any markdown formatting, remove it before considering your output complete."""
}

### Helper functions

In [5]:
def parse_json_str(json_str):
    return json.loads(json_str)

def fill_in_constants(input_str):
    for key in constants:
        input_str = input_str.replace("{"+key+"}", constants[key])
    return input_str

In [6]:
# post-processing function to remove everything before the first square bracket and after the last square bracket
def post_process(output_str):
    output_str = output_str[output_str.find("["):]
    output_str = output_str[:output_str.rfind("]") + 1]
    return output_str

### Create signature for topic classification

In [74]:
# sig_description = """Please classify the inputted question into pre-determined topics. 
# Output a subset of topics from the inputted list of topics that the question fits into. 
# Do not create new topics that aren't in the original inputted topics list.
# Return an empty string (whitespace) if the question doesn't fit into any of the inputted topics."""

sig_description = """Please think step-by-step and follow these instructions carefully.

1. Read the inputted question and list of topics.
2. For each topic in the list, determine if the question text explicitly connects with the topic. Do not make assumptions or inferences about whether the question or responses to the question are related to a topic.
3. If the question fits into a topic, add that topic to the list of topics that the question is connected with.
4. Remove all topics that are not in the original inputted topics list.
5. Return the list of topics that the question is connected with. If the question doesn't fit into any of the inputted topics, return None.
"""

# the input and output descriptions (will be read from google sheets)

input_descriptions = """{"question": "The question to classify. The input will be a JSON with the following structure: {question_json_format}", "topics": "A list of topics. The list is a string where each topic is separated by a semicolon."}"""

output_descriptions = """{"classified_topics": "A list of topics that the question is connected with. The list should be a string where each topic is separated by a semicolon. Do not number the items in the list. If the question doesn't fit into any of the inputted topics, return None."}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class ClassifyTopics(dspy.Signature):

    question = dspy.InputField(desc=fill_in_constants(input_descriptions_json["question"]))
    topics = dspy.InputField(desc=fill_in_constants(input_descriptions_json["topics"]))
    classified_topics = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["classified_topics"]))

# set the signature description
ClassifyTopics.__doc__ = sig_description

print(ClassifyTopics.__doc__)

Please think step-by-step and follow these instructions carefully.

1. Read the inputted question and list of topics.
2. For each topic in the list, determine if the question text explicitly connects with the topic. Do not make assumptions or inferences about whether the question or responses to the question are related to a topic.
3. If the question fits into a topic, add that topic to the list of topics that the question is connected with.
4. Remove all topics that are not in the original inputted topics list.
5. Return the list of topics that the question is connected with. If the question doesn't fit into any of the inputted topics, return None.



In [75]:
# Create a module
class ClassifyTopicsModule(dspy.Module):
    def __init__(self):

        super().__init__()
        
        self.classified_topics = dspy.ChainOfThought(ClassifyTopics)

    def forward(self, question, topics, return_rationale=False, temp=0.7):

        output = self.classified_topics(question=question, topics=topics, config=dict(temperature=temp))

        # return the output as a dictionary

        if return_rationale:
            return {"classified_topics": output.classified_topics, "rationale": output.rationale}
        else:
            return {"classified_topics": output.classified_topics}

In [81]:
# test_question = {
#     "cell_type": "question",
#     "response_format": "open",
#     "description": "",
#     "main_text": "What is a favorite experience or memory you have at X Park?",
#     "response_categories": []
# }

# test_question = {
#     "cell_type": "question",
#     "response_format": "open",
#     "description": "",
#     "main_text": "How often do you visit X Park?",
#     "response_categories": []
# }

test_question = {
    "cell_type": "question",
    "response_format": "open",
    "description": "",
    "main_text": "What color is the sky?",
    "response_categories": []
}

test_question_str = json.dumps(test_question)

test_topics = "Favorite Experience at X Park; Negative Experience at X Park"

In [82]:
# Test out ClassifyTopicsModule
classify_topics_module = ClassifyTopicsModule()

# Run the test input
output = classify_topics_module(question=test_question_str, topics=test_topics, temp=0.7001)

print(output)

{'classified_topics': 'None'}


In [78]:
gpt3_turbo.inspect_history(n=1)





Please think step-by-step and follow these instructions carefully.

1. Read the inputted question and list of topics.
2. For each topic in the list, determine if the question text explicitly connects with the topic. Do not make assumptions or inferences about whether the question or responses to the question are related to a topic.
3. If the question fits into a topic, add that topic to the list of topics that the question is connected with.
4. Remove all topics that are not in the original inputted topics list.
5. Return the list of topics that the question is connected with. If the question doesn't fit into any of the inputted topics, return None.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }



### Create signature for topic detection

In [48]:
sig_description = """You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

Please take the inputted context. Read and re-read the context carefully. Then, please identify a small set of mutually exclusive topics that a community survey or interview guide should touch upon, based on the inputted context. You should return a list in the requested format where each item is a topic, and where each topic is described concisely in as few characters as possible. Topics should be separated by semicolons. Do not number the items in the list."""

# description that asks for descriptions

# sig_description = """You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

# Please take the inputted context. Read and re-read the context carefully. Then, please identify a small set of mutually exclusive topics that a community survey or interview guide should touch upon, based on the inputted context. You should return a list in the requested format where each item is a topic, and where each topic is described concisely in as few characters as possible. Each topic should be followed by a colon and a short description for users to understand what the topic represents."""

# the input and output descriptions (will be read from google sheets)

input_descriptions = """{"context": "The context provided by the user. The context is organized by sections. Each section starts with three hashtag characters (###) followed by a question."}"""

output_descriptions = """{"topics": "A list of topics and descriptions. The list should be a string where each topic is separated by a semicolon. Do not number the items in the list."}"""

# output_descriptions = """{"topics": "A list of topics and descriptions. The list should be a string where each topic is separated by a newline. Do not number the items in the list."}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class DetectTopics(dspy.Signature):

    context = dspy.InputField(desc=fill_in_constants(input_descriptions_json["context"]))
    topics = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["topics"]))

# set the signature description
DetectTopics.__doc__ = sig_description

print(DetectTopics.__doc__)

You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

Please take the inputted context. Read and re-read the context carefully. Then, please identify a small set of mutually exclusive topics that a community survey or interview guide should touch upon, based on the inputted context. You should return a list in the requested format where each item is a topic, and where each topic is described concisely in as few characters as possible. Topics should be separated by semicolons. Do not number the items in the list.


In [49]:
# Create a module
class DetectTopicsModule(dspy.Module):
    def __init__(self):

        super().__init__()
        
        self.topics = dspy.ChainOfThought(DetectTopics)

    def forward(self, context, return_rationale=False, temp=0.7):

        output = self.topics(context=context, config=dict(temperature=temp))

        # return the output as a dictionary

        if return_rationale:
            return {"topics": output.topics, "rationale": output.rationale}
        else:
            return {"topics": output.topics}

In [9]:
test_input_interview = """### What is the problem to be solved or the decision to be made?
The Parks Department ("PD") of a relatively small Massachusetts city (“Freeburg”) was recently granted state funds to make improvements to local parks. The PD doesn’t often receive grants of this size, so they want to make sure they use the funds effectively; if they use all the funds, they may be eligible for another grant next year.

Freeburg has 13 parks. Some are quite small, and would only require minimal improvements (e.g., tree-planting, de-weeding), whereas others will require major improvements to address safety and usability concerns.

Parks in wealthier neighborhoods of Freeburg tend to be nicer, which some residents believe may reflect a discrepancy in how tax funds are used and distributed by the city. The residents who take issue with these distributions tend to be lower-income and tend to live farther away from these parks, which have “higher-class” amenities, like tennis courts, public bathrooms, and water fountains with ground-level dog-bowl attachments. 

### What information is needed from the public to make the decision?
The parks were once well-kept, but in recent years, have been in a state of disarray, reflecting economic challenges that hit Freeburg hard during the COVID-19 pandemic. The PD needs to interview residents of the Freeburg community to understand their needs, interests, and priorities as they relate to the local parks; this information will be used to inform what kinds of improvements are made to the parks.

The PD acknowledges that some improvements made by grant funds may lead to downstream costs that would not be covered by the grant, but the PD wishes to explore these anyway, due to their impact and long-term value for community members. For example, installing stationary trash and recycling bins in each park will help to reduce litter and improve the health and safety of the parks. However, while the state grant would pay for these bins to be installed, they would not pay for any future repairs or replacements, nor would they pay for the bins to be emptied regularly, which would be the task of the local Waste Management ("WM") service maintained by the city.

### What region is the engagement focused on? (e.g., city, county, state, national, etc.)
Freeburg

### Is the region urban, suburban, or rural? 
Suburban

### What groups of people will be affected by the outcome of the decision?
Some of the parks sit on the line with a nearby municipality, whose residents often use the parks. This may be viewed as either a challenge or opportunity by Freeburg residents, who may wish for the improved parks to be kept for their own private use, or who may wish for the parks to be shared (as they have been in the past) to expand the kinds of activities that the parks may host (for example, elementary school sporting events). 

There are several groups of constituents in the city, marked by demographic and geographic differences. Freeburg has a lower-altitude downtown (“DT”) that tends to have lower-income housing, in part due to historically long-standing social divisions, and in part due to the relatively high rate of flooding. The DT area has most of the city’s parks, but they tend to be far worse in quality, and are commonly policed (to many residents’ discomfort) to mitigate perceived issues with crime, which may or may not be the case. The DT area houses about 70% of the city’s residents, who are primarily from minority backgrounds. Freeburg also has a higher-altitude uptown (“UT”) area, whose residents tend to be higher-income. The UT area is the city’s financial and commerce district; as such, it brings in more out-of-city tourism and houses more of the city’s long-standing shopping (e.g., malls) entertainment venues (e.g., movie theaters). Residents of Freeburg are also divided by language. About 40% of the city’s residents are primarily Spanish-speaking, 8% are primarily Haitian-speaking, and 52% are primarily English-speaking. Throughout the city, signage (specifically, the languages used on public signage, such as those placed on parks) are an ongoing problem.

### Which of these groups are you engaging?
We will engage with residents in both the lower-altitude downtown (“DT”) and higher-altitude uptown (“UT”) areas.

### What form of engagement (e.g., virtual convenings, one-on-one interviews, focus groups, surveys) will best solicit the input needed from the communities you hope to engage?
One-on-one semi-structured interviews

### What is the maximum amount of time in minutes you can expect people to spend on the engagement? (e.g., 5 minutes, 30 minutes, 60 minutes, etc.)
60 minutes maximum

### What is the breakdown of open-ended and close-ended questions?
80 percent of questions are open-ended and the remaining are close-ended"""

In [10]:
test_input_survey = """### What is the problem to be solved or the decision to be made?
The Parks Department ("PD") of a relatively small Massachusetts city (“Freeburg”) was recently granted state funds to make improvements to local parks. The PD doesn’t often receive grants of this size, so they want to make sure they use the funds effectively; if they use all the funds, they may be eligible for another grant next year.

Freeburg has 13 parks. Some are quite small, and would only require minimal improvements (e.g., tree-planting, de-weeding), whereas others will require major improvements to address safety and usability concerns.

Parks in wealthier neighborhoods of Freeburg tend to be nicer, which some residents believe may reflect a discrepancy in how tax funds are used and distributed by the city. The residents who take issue with these distributions tend to be lower-income and tend to live farther away from these parks, which have “higher-class” amenities, like tennis courts, public bathrooms, and water fountains with ground-level dog-bowl attachments. 

### What information is needed from the public to make the decision?
The parks were once well-kept, but in recent years, have been in a state of disarray, reflecting economic challenges that hit Freeburg hard during the COVID-19 pandemic. The PD needs to survey the residents of the Freeburg community to understand their needs, interests, and priorities as they relate to the local parks; this information will be used to inform what kinds of improvements are made to the parks.

The PD acknowledges that some improvements made by grant funds may lead to downstream costs that would not be covered by the grant, but the PD wishes to explore these anyway, due to their impact and long-term value for community members. For example, installing stationary trash and recycling bins in each park will help to reduce litter and improve the health and safety of the parks. However, while the state grant would pay for these bins to be installed, they would not pay for any future repairs or replacements, nor would they pay for the bins to be emptied regularly, which would be the task of the local Waste Management ("WM") service maintained by the city.

### What region is the engagement focused on? (e.g., city, county, state, national, etc.)
Freeburg

### Is the region urban, suburban, or rural? 
Suburban

### What groups of people will be affected by the outcome of the decision?
Some of the parks sit on the line with a nearby municipality, whose residents often use the parks. This may be viewed as either a challenge or opportunity by Freeburg residents, who may wish for the improved parks to be kept for their own private use, or who may wish for the parks to be shared (as they have been in the past) to expand the kinds of activities that the parks may host (for example, elementary school sporting events). 

There are several groups of constituents in the city, marked by demographic and geographic differences. Freeburg has a lower-altitude downtown (“DT”) that tends to have lower-income housing, in part due to historically long-standing social divisions, and in part due to the relatively high rate of flooding. The DT area has most of the city’s parks, but they tend to be far worse in quality, and are commonly policed (to many residents’ discomfort) to mitigate perceived issues with crime, which may or may not be the case. The DT area houses about 70% of the city’s residents, who are primarily from minority backgrounds. Freeburg also has a higher-altitude uptown (“UT”) area, whose residents tend to be higher-income. The UT area is the city’s financial and commerce district; as such, it brings in more out-of-city tourism and houses more of the city’s long-standing shopping (e.g., malls) entertainment venues (e.g., movie theaters). Residents of Freeburg are also divided by language. About 40% of the city’s residents are primarily Spanish-speaking, 8% are primarily Haitian-speaking, and 52% are primarily English-speaking. Throughout the city, signage (specifically, the languages used on public signage, such as those placed on parks) are an ongoing problem.

### Which of these groups are you engaging?
We will engage with residents in both the lower-altitude downtown (“DT”) and higher-altitude uptown (“UT”) areas.

### What form of engagement (e.g., virtual convenings, one-on-one interviews, focus groups, surveys) will best solicit the input needed from the communities you hope to engage?
Online survey

### What is the maximum amount of time in minutes you can expect people to spend on the engagement? (e.g., 5 minutes, 30 minutes, 60 minutes, etc.)
10 minutes maximum

### What is the breakdown of open-ended and close-ended questions?
20 percent of questions are open-ended and the remaining are close-ended"""

In [51]:
# Test out CreateDraftModule
# Create a CreateDraftModule object
detect_topics_module = DetectTopicsModule()

# Run the test input
output = detect_topics_module(context=test_input_survey, temp=0.7003)
# output = detect_topics_module(context=test_input_interview, temp=0.7001)

print(output)

{'topics': 'Park improvement priorities; Equity in park distribution; Community demographics and needs.'}


In [35]:
gpt3_turbo.inspect_history(n=1)





You are an advanced survey and interview guide builder designed to use the information provided by users to create surveys and interview guides that they can use for their constituents. 

Please take the inputted context. Read and re-read the context carefully. Then, please identify a set of mutually exclusive topics that a community survey or interview guide should touch upon, based on the inputted context. You should return a list in the requested format where each item is a topic, and where each topic is described concisely in as few characters as possible. Each topic should be followed by a colon and a short description for users to understand what the topic represents.

---

Follow the following format.

Context: The context provided by the user. The context is organized by sections. Each section starts with three hashtag characters (###) followed by a question.
Reasoning: Let's think step by step in order to ${produce the topics}. We ...
Topics: A list of topics and descripti

### Create signature for step 2

In [6]:
# Create a class-based DSPy Signature to sgenerate first draft of questions  

# the signature description (will be read from google sheets)

# original description that Danny created
# original_sig_description = """ROLE: You are an advanced survey builder designed to use the information provided by users to create streamlined surveys that they can use for their constituents. 

# INSTRUCTIONS: Please think step-by-step. 
# 1. The user has provided the following context. Please read and re-read the context carefully: {CONTEXT}. 
# 2. Please generate of questions to help the user elicit useful information from their constituents. There should be roughly [X] number of questions, where [Y] percent are open ended and the remaining are close ended. 
# 3. Return a json list in this JSON format: 
# {
#         "response_format": "open" or "closed",
#         "description": string,
#         "main_text": string,
#         "rationale": string,
#         "response_categories": empty list or list of JSONs with an "id" and "text" field
#     }
# 4. Please review the question provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your response between 20 and 50 words. Use semicolons to separate list items. Please start your response with "This question" followed by your rationale. For example: "This question is being asked in order to...""""

sig_description = """ROLE: You are an advanced survey and interview guide builder designed to use the information provided by users to create streamlined surveys and interview guides that they can use for their constituents.

INSTRUCTIONS: Please think step-by-step.
1. The user has provided some context. Please read and re-read the inputted context carefully. 
2. Please generate a list of questions to help the user elicit useful information from their constituents. The total number of questions should be compatible with the time limit specified in the context. The proportion of open and closed questions is also defined in the context.
3. Please organize the questions into sections. Each section should have a title and time estimate in minutes.
4. Please review the questions provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your response between 20 and 50 words. Use semicolons to separate list items. Please start your response with "This question" followed by your rationale. For example: "This question is being asked in order to... These rationales should populate the "rationale" field in the JSON output."""

# the input and output descriptions (will be read from google sheets)

input_descriptions = """{"context": "The context provided by the user. The context is organized by sections. Each section starts with three hashtag characters (###) followed by a question."}"""

output_descriptions = """{"questions": "The sections and questions. The output should be a list of JSONs where each element has the following structure: {section_json_format}. {json_formatting_message}"}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class CreateDraft(dspy.Signature):

    context = dspy.InputField(desc=fill_in_constants(input_descriptions_json["context"]))
    questions = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["questions"]))

# set the signature description
CreateDraft.__doc__ = sig_description

print(CreateDraft.__doc__)

ROLE: You are an advanced survey and interview guide builder designed to use the information provided by users to create streamlined surveys and interview guides that they can use for their constituents.

INSTRUCTIONS: Please think step-by-step.
1. The user has provided some context. Please read and re-read the inputted context carefully. 
2. Please generate a list of questions to help the user elicit useful information from their constituents. The total number of questions should be compatible with the time limit specified in the context. The proportion of open and closed questions is also defined in the context.
3. Please organize the questions into sections. Each section should have a title and time estimate in minutes.
4. Please review the questions provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your response between 20 and 50 words. Use semicolons to separate list items. Please start your response wit

In [110]:
interview_sig_description = """<role>
You are an advanced interview guide builder designed to use the information provided by users to create interview guides that they can use for their constituents.
</role>

<instructions>
Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided previously.
2. Please generate a list of questions to help the user elicit useful information from their constituents. The total number of questions should be compatible with the time limit specified in the context. The proportion of open and closed questions is also defined in the context.
3. Please organize the questions into sections. Each section should have a title and time estimate in minutes. The questions should progress logically and linearly through a <beginning>, <middle>, and <end> to form a full interview guide. 
4. Please review the questions provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your response between 20 and 50 words. Use semicolons to separate list items. Please start your response with “This question” followed by your rationale. For example: “This question is being asked in order to... These rationales should populate the “rationale” field in the JSON output.
5. Please generate a title for the interview guide. The title should be brief yet descriptive, accurately capturing the core context that is being explored through this interview endeavor. 
6. Ensure that your output excludes any references to the instructions provided. For example, exclude reference to <beginning>, <middle>, or <end>.

<beginning>
To construct the first few questions in your interview guide, think step-by-step and follow these instructions carefully.
1. Provide initial context on the interview, which includes the purpose, length, the nature of the interview, and how the information will be used.
2. Start with an easy-to-answer, open question that is non threatening. The question should ask for information that helps the interviewer frame the next part of the interview.
</beginning>

<middle>
To construct the next few questions in your interview guide, abide by these principles:
Principle 1. Start with concrete, accessible and easy-to-answer questions about experiences Later, when the respondent is at ease and a degree of rapport is established, it is more fruitful to ask more challenging and abstract questions.
Principle 2. The middle should consist of a short list of core questions with possible probing questions under each core question.
Principle 3. Probing questions prompt respondents to reflect on, explain, and modify initial statements. Clearinghouse probes make sure you got all the important information about a topic by encouraging respondents to volunteer additional information. Informational probes ask for additional information or explanation (e.g., “Why do you think that…” or “What do you think…“).
Principle 4. Strive for mostly open-ended questions with a few close-ended questions sprinkled throughout as breaks for respondents. Open-ended questions can start with the following stems: “Tell me about”, “Where were you when”, “Who was with you when”, What happened after”, “What did you say or do when”, “How did you feel when”, “What reasons did you have for”.
</middle>

<end>
To construct the final few questions in your interview guide, think step-by-step and follow these instructions carefully.
1. Use a clearinghouse question. Examples are “what have I not asked that you think is important for me to know?” or “have I answered all of your questions?”
2. Express appreciation or satisfaction. Examples are “I really appreciate you making time for me on such a busy time” or “Thanks for being so candid with me”.
3. Lay the groundwork for future contact. Explain what will happen next, where it will happen, when it will happen, and why it will happen.
</end>
</instructions>"""

# the input and output descriptions (will be read from google sheets)

interview_input_descriptions = """{"context": "The context provided by the user. The context is organized by sections. Each section starts with three hashtag characters (###) followed by a question."}"""

interview_output_descriptions = """{"questions": "The sections and questions. The output should be a list of JSON objects enclosed in square brackets, with each object separated by a comma. Each JSON object should have the following structure: {section_json_format}.", "title": "The title of the interview guide as a string."}"""

# convert the descriptions to JSON
interview_input_descriptions_json = parse_json_str(interview_input_descriptions)
interview_output_descriptions_json = parse_json_str(interview_output_descriptions)

class CreateInterviewDraft(dspy.Signature):

    context = dspy.InputField(desc=fill_in_constants(interview_input_descriptions_json["context"]))
    questions = dspy.OutputField(desc=fill_in_constants(interview_output_descriptions_json["questions"]))
    title = dspy.OutputField(desc=fill_in_constants(interview_output_descriptions_json["title"]))

# set the signature description
CreateInterviewDraft.__doc__ = interview_sig_description

print(CreateInterviewDraft.__doc__)

<role>
You are an advanced interview guide builder designed to use the information provided by users to create interview guides that they can use for their constituents.
</role>

<instructions>
Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided previously.
2. Please generate a list of questions to help the user elicit useful information from their constituents. The total number of questions should be compatible with the time limit specified in the context. The proportion of open and closed questions is also defined in the context.
3. Please organize the questions into sections. Each section should have a title and time estimate in minutes. The questions should progress logically and linearly through a <beginning>, <middle>, and <end> to form a full interview guide. 
4. Please review the questions provided and compose a detailed explanation for why that question is being asked of members of the given commu

In [119]:
survey_sig_description = """<role>
You are an advanced survey builder designed to use the information provided by users to create surveys that they can use for their constituents.
</role>

<instructions>
Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided previously.
2. Please generate a list of questions to help the user elicit useful information from their constituents. The total number of questions should be compatible with the time limit specified in the context. The proportion of open and closed questions is also defined in the context. 
3. Please organize the questions into sections. Each section should have a title and time estimate in minutes. The questions should progress logically and linearly through a <beginning>, <middle>, and <end> to form a full survey. 
4. Please review the questions provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your response between 20 and 50 words. Use semicolons to separate list items. Please start your response with “This question” followed by your rationale. For example: “This question is being asked in order to... These rationales should populate the “rationale” field in the JSON output.
5. Please generate a title for the survey. The title should be brief yet descriptive, accurately capturing the core context that is being explored through this survey endeavor. 
6. Ensure that your output excludes any references to the instructions provided. For example, exclude reference to <beginning>, <middle>, or <end>.

<beginning>
To construct the first few questions in your survey, think step-by-step and follow these instructions carefully.
1. Provide initial context on the survey, which includes the purpose and length.
2. Specify how the information will be used in a way that highlights possible benefits of the survey for the community and respondent themselves.
3. Assure respondents of complete anonymity.
</beginning>

<middle>
To construct the next few questions in your survey, abide by these principles:
Principle 1. It’s usually best to start a survey with general questions that will be easy for a respondent to answer.
Principle 2. It is best to organize the survey logically and to guide respondents without jumping from one topic to another, which can irritate respondents.
Principle 3. Denote changes in topic with a new section. Also introduce new topics and explain why they are being asked.
Principle 4. Things mentioned early in a survey can impact answers later, so care must be taken when ordering the questions.
Principle 5. Keep the survey short. Respondents are less likely to answer a long questionnaire than a short one, and often pay less attention to questionnaires which seem long, monotonous, or boring.
Principle 6. Include both open-ended and closed-ended questions. But use open-ended questions sparingly and place them earlier on in the survey while respondents are less tired.
</middle>

<end>
To construct the final few questions in your survey, think step-by-step and follow these instructions carefully.
1. It’s usually best to ask any sensitive questions, including demographics (especially income), near the end of the survey
2. Use a clearinghouse question. Example is “Is there any additional context you’d like to share on your responses to the previous questions, or anything else you’d like to share?”
3. Lay the groundwork for future contact. Explain what will happen next, where it will happen, when it will happen, and why it will happen.
</end>
</instructions>"""

# the input and output descriptions (will be read from google sheets)

survey_input_descriptions = """{"context": "The context provided by the user. The context is organized by sections. Each section starts with three hashtag characters (###) followed by a question."}"""

survey_output_descriptions = """{"questions": "The sections and questions. The output should be a list of JSON objects enclosed in square brackets, with each object separated by a comma. Each JSON object should have the following structure: {section_json_format}.", "title": "The title of the survey as a string."}"""

# convert the descriptions to JSON
survey_input_descriptions_json = parse_json_str(survey_input_descriptions)
survey_output_descriptions_json = parse_json_str(survey_output_descriptions)

class CreateSurveyDraft(dspy.Signature):

    context = dspy.InputField(desc=fill_in_constants(survey_input_descriptions_json["context"]))
    questions = dspy.OutputField(desc=fill_in_constants(survey_output_descriptions_json["questions"]))
    title = dspy.OutputField(desc=fill_in_constants(survey_output_descriptions_json["title"]))

# set the signature description
CreateSurveyDraft.__doc__ = survey_sig_description

print(CreateSurveyDraft.__doc__)

<role>
You are an advanced survey builder designed to use the information provided by users to create surveys that they can use for their constituents.
</role>

<instructions>
Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided previously.
2. Please generate a list of questions to help the user elicit useful information from their constituents. The total number of questions should be compatible with the time limit specified in the context. The proportion of open and closed questions is also defined in the context. 
3. Please organize the questions into sections. Each section should have a title and time estimate in minutes. The questions should progress logically and linearly through a <beginning>, <middle>, and <end> to form a full survey. 
4. Please review the questions provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your res

In [120]:
# Create a module
class CreateDraftModule(dspy.Module):
    def __init__(self):

        super().__init__()

        self.interview_draft = dspy.Predict(CreateInterviewDraft)
        self.survey_draft = dspy.Predict(CreateSurveyDraft)

    def forward(self, context, draft_type="interview", temp=0.7):

        if draft_type == "interview":
            output = self.interview_draft(context=context, config=dict(temperature=temp))
            # return the output as a dictionary
            return {"title": output.title, "sections": output.questions}
        elif draft_type == "survey":
            output = self.survey_draft(context=context, config=dict(temperature=temp))
            # return the output as a dictionary
            return {"title": output.title, "sections": output.questions}
        else:
            raise ValueError("Invalid draft type. Please choose either 'interview' or 'survey'.")

        

In [112]:
test_input_interview = """### What is the problem to be solved or the decision to be made?
The Parks Department ("PD") of a relatively small Massachusetts city (“Freeburg”) was recently granted state funds to make improvements to local parks. The PD doesn’t often receive grants of this size, so they want to make sure they use the funds effectively; if they use all the funds, they may be eligible for another grant next year.

Freeburg has 13 parks. Some are quite small, and would only require minimal improvements (e.g., tree-planting, de-weeding), whereas others will require major improvements to address safety and usability concerns.

Parks in wealthier neighborhoods of Freeburg tend to be nicer, which some residents believe may reflect a discrepancy in how tax funds are used and distributed by the city. The residents who take issue with these distributions tend to be lower-income and tend to live farther away from these parks, which have “higher-class” amenities, like tennis courts, public bathrooms, and water fountains with ground-level dog-bowl attachments. 

### What information is needed from the public to make the decision?
The parks were once well-kept, but in recent years, have been in a state of disarray, reflecting economic challenges that hit Freeburg hard during the COVID-19 pandemic. The PD needs to interview residents of the Freeburg community to understand their needs, interests, and priorities as they relate to the local parks; this information will be used to inform what kinds of improvements are made to the parks.

The PD acknowledges that some improvements made by grant funds may lead to downstream costs that would not be covered by the grant, but the PD wishes to explore these anyway, due to their impact and long-term value for community members. For example, installing stationary trash and recycling bins in each park will help to reduce litter and improve the health and safety of the parks. However, while the state grant would pay for these bins to be installed, they would not pay for any future repairs or replacements, nor would they pay for the bins to be emptied regularly, which would be the task of the local Waste Management ("WM") service maintained by the city.

### What region is the engagement focused on? (e.g., city, county, state, national, etc.)
Freeburg

### Is the region urban, suburban, or rural? 
Suburban

### What groups of people will be affected by the outcome of the decision?
Some of the parks sit on the line with a nearby municipality, whose residents often use the parks. This may be viewed as either a challenge or opportunity by Freeburg residents, who may wish for the improved parks to be kept for their own private use, or who may wish for the parks to be shared (as they have been in the past) to expand the kinds of activities that the parks may host (for example, elementary school sporting events). 

There are several groups of constituents in the city, marked by demographic and geographic differences. Freeburg has a lower-altitude downtown (“DT”) that tends to have lower-income housing, in part due to historically long-standing social divisions, and in part due to the relatively high rate of flooding. The DT area has most of the city’s parks, but they tend to be far worse in quality, and are commonly policed (to many residents’ discomfort) to mitigate perceived issues with crime, which may or may not be the case. The DT area houses about 70% of the city’s residents, who are primarily from minority backgrounds. Freeburg also has a higher-altitude uptown (“UT”) area, whose residents tend to be higher-income. The UT area is the city’s financial and commerce district; as such, it brings in more out-of-city tourism and houses more of the city’s long-standing shopping (e.g., malls) entertainment venues (e.g., movie theaters). Residents of Freeburg are also divided by language. About 40% of the city’s residents are primarily Spanish-speaking, 8% are primarily Haitian-speaking, and 52% are primarily English-speaking. Throughout the city, signage (specifically, the languages used on public signage, such as those placed on parks) are an ongoing problem.

### Which of these groups are you engaging?
We will engage with residents in both the lower-altitude downtown (“DT”) and higher-altitude uptown (“UT”) areas.

### What form of engagement (e.g., virtual convenings, one-on-one interviews, focus groups, surveys) will best solicit the input needed from the communities you hope to engage?
One-on-one semi-structured interviews

### What is the maximum amount of time in minutes you can expect people to spend on the engagement? (e.g., 5 minutes, 30 minutes, 60 minutes, etc.)
60 minutes maximum

### What is the breakdown of open-ended and close-ended questions?
80 percent of questions are open-ended and the remaining are close-ended"""

In [121]:
test_input_survey = """### What is the problem to be solved or the decision to be made?
The Parks Department ("PD") of a relatively small Massachusetts city (“Freeburg”) was recently granted state funds to make improvements to local parks. The PD doesn’t often receive grants of this size, so they want to make sure they use the funds effectively; if they use all the funds, they may be eligible for another grant next year.

Freeburg has 13 parks. Some are quite small, and would only require minimal improvements (e.g., tree-planting, de-weeding), whereas others will require major improvements to address safety and usability concerns.

Parks in wealthier neighborhoods of Freeburg tend to be nicer, which some residents believe may reflect a discrepancy in how tax funds are used and distributed by the city. The residents who take issue with these distributions tend to be lower-income and tend to live farther away from these parks, which have “higher-class” amenities, like tennis courts, public bathrooms, and water fountains with ground-level dog-bowl attachments. 

### What information is needed from the public to make the decision?
The parks were once well-kept, but in recent years, have been in a state of disarray, reflecting economic challenges that hit Freeburg hard during the COVID-19 pandemic. The PD needs to survey the residents of the Freeburg community to understand their needs, interests, and priorities as they relate to the local parks; this information will be used to inform what kinds of improvements are made to the parks.

The PD acknowledges that some improvements made by grant funds may lead to downstream costs that would not be covered by the grant, but the PD wishes to explore these anyway, due to their impact and long-term value for community members. For example, installing stationary trash and recycling bins in each park will help to reduce litter and improve the health and safety of the parks. However, while the state grant would pay for these bins to be installed, they would not pay for any future repairs or replacements, nor would they pay for the bins to be emptied regularly, which would be the task of the local Waste Management ("WM") service maintained by the city.

### What region is the engagement focused on? (e.g., city, county, state, national, etc.)
Freeburg

### Is the region urban, suburban, or rural? 
Suburban

### What groups of people will be affected by the outcome of the decision?
Some of the parks sit on the line with a nearby municipality, whose residents often use the parks. This may be viewed as either a challenge or opportunity by Freeburg residents, who may wish for the improved parks to be kept for their own private use, or who may wish for the parks to be shared (as they have been in the past) to expand the kinds of activities that the parks may host (for example, elementary school sporting events). 

There are several groups of constituents in the city, marked by demographic and geographic differences. Freeburg has a lower-altitude downtown (“DT”) that tends to have lower-income housing, in part due to historically long-standing social divisions, and in part due to the relatively high rate of flooding. The DT area has most of the city’s parks, but they tend to be far worse in quality, and are commonly policed (to many residents’ discomfort) to mitigate perceived issues with crime, which may or may not be the case. The DT area houses about 70% of the city’s residents, who are primarily from minority backgrounds. Freeburg also has a higher-altitude uptown (“UT”) area, whose residents tend to be higher-income. The UT area is the city’s financial and commerce district; as such, it brings in more out-of-city tourism and houses more of the city’s long-standing shopping (e.g., malls) entertainment venues (e.g., movie theaters). Residents of Freeburg are also divided by language. About 40% of the city’s residents are primarily Spanish-speaking, 8% are primarily Haitian-speaking, and 52% are primarily English-speaking. Throughout the city, signage (specifically, the languages used on public signage, such as those placed on parks) are an ongoing problem.

### Which of these groups are you engaging?
We will engage with residents in both the lower-altitude downtown (“DT”) and higher-altitude uptown (“UT”) areas.

### What form of engagement (e.g., virtual convenings, one-on-one interviews, focus groups, surveys) will best solicit the input needed from the communities you hope to engage?
Online survey

### What is the maximum amount of time in minutes you can expect people to spend on the engagement? (e.g., 5 minutes, 30 minutes, 60 minutes, etc.)
10 minutes maximum

### What is the breakdown of open-ended and close-ended questions?
20 percent of questions are open-ended and the remaining are close-ended"""

In [122]:
# Test out CreateDraftModule
# Create a CreateDraftModule object
create_draft_module = CreateDraftModule()

# Run the test input
# output = create_draft_module(context=test_input_interview, draft_type="interview", temp=0.7001)
output = create_draft_module(context=test_input_survey, draft_type="survey", temp=0.7001)

print(output)

{'sections': '```json\n[\n  {\n    "id": 0,\n    "title": "Introduction and Purpose",\n    "time_estimate": 1,\n    "cells": [\n      {\n        "cell_type": "text",\n        "response_format": "open",\n        "main_text": "Welcome to the Freeburg Parks Improvement Survey. The purpose of this survey is to gather your valuable input on how the recent state funds should be utilized to enhance our local parks. Your feedback will directly influence the improvements we make, aiming to benefit our community and possibly secure future funding for continued enhancements. Please be assured that your responses are completely anonymous.",\n        "rationale": "This text provides respondents with the context and purpose of the survey, ensuring they understand the importance of their participation; it also reassures them about their privacy."\n      }\n    ]\n  },\n  {\n    "id": 1,\n    "title": "General Park Usage",\n    "time_estimate": 3,\n    "cells": [\n      {\n        "cell_type": "questi

In [114]:
# get the number of tokens
num_tokens = num_tokens_from_string(output["sections"], "gpt-4-0125-preview")

print(f"Number of tokens: {num_tokens}")

Number of tokens: 1126


In [123]:
# let's see if output.sections is a json
try:
    json_output = json.loads(post_process(output["sections"]))
    # print json_output in a nice way
    print(json.dumps(json_output, indent=4))
    print("output is json")
except:
    print("output is not json")

[
    {
        "id": 0,
        "title": "Introduction and Purpose",
        "time_estimate": 1,
        "cells": [
            {
                "cell_type": "text",
                "response_format": "open",
                "main_text": "Welcome to the Freeburg Parks Improvement Survey. The purpose of this survey is to gather your valuable input on how the recent state funds should be utilized to enhance our local parks. Your feedback will directly influence the improvements we make, aiming to benefit our community and possibly secure future funding for continued enhancements. Please be assured that your responses are completely anonymous.",
                "rationale": "This text provides respondents with the context and purpose of the survey, ensuring they understand the importance of their participation; it also reassures them about their privacy."
            }
        ]
    },
    {
        "id": 1,
        "title": "General Park Usage",
        "time_estimate": 3,
        "cel

In [125]:
gpt4.inspect_history(n=1)





<role>
You are an advanced survey builder designed to use the information provided by users to create surveys that they can use for their constituents.
</role>

<instructions>
Please think step-by-step and follow these instructions carefully.
1. Please read and re-read the context that the user has provided previously.
2. Please generate a list of questions to help the user elicit useful information from their constituents. The total number of questions should be compatible with the time limit specified in the context. The proportion of open and closed questions is also defined in the context. 
3. Please organize the questions into sections. Each section should have a title and time estimate in minutes. The questions should progress logically and linearly through a <beginning>, <middle>, and <end> to form a full survey. 
4. Please review the questions provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your

In [53]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





ROLE: You are an advanced survey and interview guide builder designed to use the information provided by users to create streamlined surveys and interview guides that they can use for their constituents.

INSTRUCTIONS: Please think step-by-step.
1. The user has provided some context. Please read and re-read the inputted context carefully. 
2. Please generate a list of questions to help the user elicit useful information from their constituents. The total number of questions should be compatible with the time limit specified in the context. The proportion of open and closed questions is also defined in the context.
3. Please organize the questions into sections. Each section should have a title and time estimate in minutes.
4. Please review the questions provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your response between 20 and 50 words. Use semicolons to separate list items. Please start your response

### Create signature for step 2

In [1]:
# Create a class-based DSPy Signature to sgenerate first draft of questions  

# the signature description (will be read from google sheets)

# original description that Danny created
# original_sig_description = """ROLE: You are an advanced survey builder designed to use the information provided by users to create streamlined surveys that they can use for their constituents. 

# INSTRUCTIONS: Please think step-by-step. 
# 1. The user has provided the following context. Please read and re-read the context carefully: {CONTEXT}. 
# 2. Please generate of questions to help the user elicit useful information from their constituents. There should be roughly [X] number of questions, where [Y] percent are open ended and the remaining are close ended. 
# 3. Return a json list in this JSON format: 
# {
#         "response_format": "open" or "closed",
#         "description": string,
#         "main_text": string,
#         "rationale": string,
#         "response_categories": empty list or list of JSONs with an "id" and "text" field
#     }
# 4. Please review the question provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your response between 20 and 50 words. Use semicolons to separate list items. Please start your response with "This question" followed by your rationale. For example: "This question is being asked in order to...""""

sig_description = """ROLE: You are an advanced survey and interview guide builder designed to use the information provided by users to create streamlined surveys and interview guides that they can use for their constituents.

INSTRUCTIONS: Please think step-by-step.
1. The user has provided some context. Please read and re-read the inputted context carefully. 
2. Please generate a list of questions to help the user elicit useful information from their constituents. The total number of questions should be compatible with the time limit specified in the context. The proportion of open and closed questions is also defined in the context.
3. Please organize the questions into sections. Each section should have a title and time estimate in minutes.
4. Please review the questions provided and compose a detailed explanation for why that question is being asked of members of the given community. Please keep your response between 20 and 50 words. Use semicolons to separate list items. Please start your response with "This question" followed by your rationale. For example: "This question is being asked in order to... These rationales should populate the "rationale" field in the JSON output."""

# the input and output descriptions (will be read from google sheets)

input_descriptions = """{"context": "The context provided by the user. The context is organized by sections. Each section starts with three hashtag characters (###) followed by a question."}"""

output_descriptions = """{"questions": "The sections and questions. The output should be a list of JSONs where each element has the following structure: {section_json_format}.
    {json_formatting_message}"}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class CreateDraft(dspy.Signature):

    context = dspy.InputField(desc=fill_in_constants(input_descriptions_json["context"]))
    questions = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["questions"]))

# set the signature description
CreateDraft.__doc__ = sig_description

print(CreateDraft.__doc__)

NameError: name 'parse_json_str' is not defined

In [ ]:
# Create a module
class CreateDraftModule(dspy.Module):
    def __init__(self):

        super().__init__()
        
        self.draft = dspy.ChainOfThought(CreateDraft)

    def forward(self, context, return_rationale=False, temp=0.7):

        output = self.draft(question=context, config=dict(temperature=temp))

        # return the output as a dictionary

        if return_rationale:
            return {"sections": output.questions, "rationale": output.rationale}
        else:
            return {"sections": output.questions}

In [ ]:
# Test out CreateDraftModule

# Create a test input
test_input = {
    "question": {
        "cell_type": "question",
        "response_format": "open",
        "description": "A question about park amenities",
        "main_text": "What kinds of amenities do you use most in local parks?",
        "response_categories": []
    }
}

test_input = {
    "question": {
        "cell_type": "question",
        "response_format": "closed",
        "description": "A question about park amenities",
        "main_text": "What kinds of amenities do you use most in local parks?",
        "response_categories": [
            {"id": 1, "text": "Playgrounds"},
            {"id": 2, "text": "Picnic areas"},
            {"id": 3, "text": "Walking trails"},
            {"id": 4, "text": "Sports fields"},
            {"id": 5, "text": "Other"}
        ]
    }
}

test_input_str = json.dumps(test_input["question"])

# Create a RewordQuestionsModule object
switch = SwitchResponseFormatModule()

# Run the test input
output = switch(question=test_input_str, return_rationale=True)

print(output)

{'new_question': '{"cell_type": "question", "response_format": "open", "description": "A question about park amenities", "main_text": "What kinds of amenities do you use most in local parks?"}', 'rationale': 'produce the new_question. We need to change the format from closed to open, so we can ask for more detailed information about park amenities.'}


In [ ]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Please switch the response format for the inputted question. Open questions should become closed, and closed questions should become open. Preserve as much of the original meaning as possible.

---

Follow the following format.

Question: The question to modify. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

Reasoning: Let's think step by step in order to ${produce the new_question}. We ...

New Question: The modified question. The output should be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

---

Question: {"cell_type": "

### Create signature for switch response format

In [5]:
# Create a class-based DSPy Signature to switch the response format 

# the signature description (will be read from google sheets)

sig_description = """Please switch the response format for the inputted question. Open questions should become closed, and closed questions should become open. Preserve as much of the original meaning as possible. """

# the input and output descriptions (will be read from google sheets)

input_descriptions = """{"question": "The question to modify. The input will be a JSON with the following structure: {question_json_format}"}"""

output_descriptions = """{"new_question": "The modified question. The output should be a JSON with the following structure: {question_json_format}"}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class SwitchResponseFormat(dspy.Signature):

    question = dspy.InputField(desc=fill_in_constants(input_descriptions_json["question"]))
    new_question = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["new_question"]))

# set the signature description
SwitchResponseFormat.__doc__ = sig_description

print(SwitchResponseFormat.__doc__)

Please switch the response format for the inputted question. Open questions should become closed, and closed questions should become open. Preserve as much of the original meaning as possible. 


In [6]:
# Create a module
class SwitchResponseFormatModule(dspy.Module):
    def __init__(self):

        super().__init__()
        
        self.new_question = dspy.ChainOfThought(SwitchResponseFormat)

    def forward(self, question, return_rationale=False, temp=0.7):

        output = self.new_question(question=question, config=dict(temperature=temp))

        # return the output as a dictionary

        if return_rationale:
            return {"new_question": output.new_question, "rationale": output.rationale}
        else:
            return {"new_question": output.new_question}

In [11]:
# Test out RewordQuestions

# Create a test input
test_input = {
    "question": {
        "cell_type": "question",
        "response_format": "open",
        "description": "A question about park amenities",
        "main_text": "What kinds of amenities do you use most in local parks?",
        "response_categories": []
    }
}

test_input = {
    "question": {
        "cell_type": "question",
        "response_format": "closed",
        "description": "A question about park amenities",
        "main_text": "What kinds of amenities do you use most in local parks?",
        "response_categories": [
            {"id": 1, "text": "Playgrounds"},
            {"id": 2, "text": "Picnic areas"},
            {"id": 3, "text": "Walking trails"},
            {"id": 4, "text": "Sports fields"},
            {"id": 5, "text": "Other"}
        ]
    }
}

test_input_str = json.dumps(test_input["question"])

# Create a RewordQuestionsModule object
switch = SwitchResponseFormatModule()

# Run the test input
output = switch(question=test_input_str, return_rationale=True)

print(output)

{'new_question': '{"cell_type": "question", "response_format": "open", "description": "A question about park amenities", "main_text": "What kinds of amenities do you use most in local parks?"}', 'rationale': 'produce the new_question. We need to change the format from closed to open, so we can ask for more detailed information about park amenities.'}


In [12]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Please switch the response format for the inputted question. Open questions should become closed, and closed questions should become open. Preserve as much of the original meaning as possible.

---

Follow the following format.

Question: The question to modify. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

Reasoning: Let's think step by step in order to ${produce the new_question}. We ...

New Question: The modified question. The output should be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

---

Question: {"cell_type": "

### Create signature for specific rewording

In [104]:
# Create a class-based DSPy Signature to generate an alternative question based on user's request

# original_prompt = """Please carefully review the following question, which will be asked of community members during an upcoming text survey. 

# The question is: "What kinds of amenities do you use most in local parks?"

# A civic leader wants to re-write the question to achieve the following request: "Make this question more focused on different activities people can do in the park"

# Please think step-by-step and follow these instructions carefully:

# 1. Read the question and user request above and identify the key differences between the question and user request.
# 2. Re-write the question in 3 ways that incorporate the differences you identified.  
# 3. Display the text of the questions, without any additional information of how or why you came to display them. For example, your response should take the form of a list of 3 questions. Remove any text relating to the reasoning of how you came about creating them. """

# the signature description (will be read from google sheets)

sig_description = """Please carefully review the inputted question, which will be asked of community members through a particular format that is also inputted.

A civic leader wants to re-write the question to achieve an inputted request.

Please think step-by-step and follow these instructions carefully:

1. Read the inputted question and request and identify the key differences between the question and user request.
2. Re-write the question in 3 ways that incorporate the differences you identified.  
3. Output the 3 reworded questions in the requested format."""

# the input and output descriptions (will be read from google sheets)

input_descriptions = """{"question": "The question to reword. The input will be a JSON with the following structure: {question_json_format}",
"format": "How community members will be asked the question. For example, a text survey or interview.",
"request": "The user request to guide the rewording process."}"""

output_descriptions = """{"rewordings": "The 3 reworded questions. The output should be a list of JSONs with the following structure: [{question_json_format}]"}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class RewordQuestionFromRequest(dspy.Signature):

    question = dspy.InputField(desc=fill_in_constants(input_descriptions_json["question"]))
    format = dspy.InputField(desc=fill_in_constants(input_descriptions_json["format"]))
    request = dspy.InputField(desc=fill_in_constants(input_descriptions_json["request"]))
    rewordings = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["rewordings"]))

# set the signature description
RewordQuestionFromRequest.__doc__ = sig_description

print(RewordQuestionFromRequest.__doc__)

Please carefully review the inputted question, which will be asked of community members through a particular format that is also inputted.

A civic leader wants to re-write the question to achieve an inputted request.

Please think step-by-step and follow these instructions carefully:

1. Read the inputted question and request and identify the key differences between the question and user request.
2. Re-write the question in 3 ways that incorporate the differences you identified.  
3. Output the 3 reworded questions in the requested format.


In [105]:
# Create a module
class RewordQuestionFromRequestModule(dspy.Module):
    def __init__(self, vary_temp=False):

        super().__init__()

        if vary_temp:
            rand_int = random.randint(1, 100)
        else:
            rand_int = 0
        
        self.rewordings = dspy.ChainOfThought(RewordQuestionFromRequest, temperature=0.7+0.0001*rand_int)

    def forward(self, question, format, request):

        # this needs to return a dict and not a string for Evaluate to work
        return self.rewordings(question=question, format=format, request=request)

In [106]:
# Test out RewordQuestions

# Create a test input
test_input = {
    "question": {
        "response_format": "open",
        "description": "A question about park amenities",
        "main_text": "What kinds of amenities do you use most in local parks?",
        "response_categories": []
    },
    "format": "text survey",
    "request": "Make this question more focused on different activities people can do in the park"
}

test_input_str = json.dumps(test_input["question"])

# Create a RewordQuestionsModule object
reword_question_from_request = RewordQuestionFromRequestModule(vary_temp=True)

# Run the test input
output = reword_question_from_request(question=test_input_str, format=test_input["format"], request=test_input["request"])
rewordings = output.rewordings
print(f"Rationale is {output.rationale}")

# try to convert to json
try:
    rewordings = json.loads(rewordings)
    print("Correct output format")
    print(json.dumps(rewordings, indent=4))
except:
    print("Wrong output format")
    print(rewordings)

Rationale is In this case, the key difference is that the original question asks about amenities in parks, while the user request is to focus on different activities people can do in the park. 

Rephrased questions:

1. 
```json
{
  "response_format": "open",
  "description": "A question about park activities",
  "main_text": "What activities do you enjoy doing most in local parks?",
  "response_categories": []
}
```

2. 
```json
{
  "response_format": "open",
  "description": "A question about park activities",
  "main_text": "Which recreational activities do you engage in the most when visiting local parks?",
  "response_categories": []
}
```

3. 
```json
{
  "response_format": "open",
  "description": "A question about park activities",
  "main_text": "What specific activities bring you the most joy when spending time in local parks?",
  "response_categories": []
}
```
Correct output format
[
    {
        "response_format": "open",
        "description": "A question about park acti

In [107]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Please carefully review the inputted question, which will be asked of community members through a particular format that is also inputted.

A civic leader wants to re-write the question to achieve an inputted request.

Please think step-by-step and follow these instructions carefully:

1. Read the inputted question and request and identify the key differences between the question and user request.
2. Re-write the question in 3 ways that incorporate the differences you identified.  
3. Output the 3 reworded questions in the requested format.

---

Follow the following format.

Question: The question to reword. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

Format: How community members will be asked the question. For example, a text survey or interview.

Request: The us

### Create signature for rewording

In [84]:
# Create a class-based DSPy Signature to generate an alternative wording for a question

# the signature description (will be read from google sheets)

sig_description = """Please carefully review the inputted question, which will be asked of community members through a particular format that is also inputted.

Please think step-by-step and follow these instructions carefully:

1. Read the inputted question and identify 3 possible issues that may limit its effectiveness. For example does the question assume cases that may not be true?
2. Of the issues you identified, identify the most immediate, likely, and realistic issue, which may limit the effectiveness of this question in informing decision-makers of how they should address the central challenge of this survey.
3. Based on the issue you selected, please write a 3 questions that may serve as superior alternatives to the one listed above, and which overcome the issue you found to be most prevalent.
4. Output the reworded questions in the requested format."""

# the input and output descriptions (will be read from google sheets)

input_descriptions = """{"question": "The question to reword. The input will be a JSON with the following structure: {question_json_format}",
"format": "How community members will be asked the question. For example, a text survey or interview."}"""

output_descriptions = """{"rewordings": "The reworded questions. The output should be a list of JSONs with the following structure: [{question_json_format}]"}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class RewordQuestion(dspy.Signature):

    question = dspy.InputField(desc=fill_in_constants(input_descriptions_json["question"]))
    format = dspy.InputField(desc=fill_in_constants(input_descriptions_json["format"]))
    rewordings = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["rewordings"]))

# set the signature description
RewordQuestion.__doc__ = sig_description

print(RewordQuestion.__doc__)

Please carefully review the inputted question, which will be asked of community members through a particular format that is also inputted.

Please think step-by-step and follow these instructions carefully:

1. Read the inputted question and identify 3 possible issues that may limit its effectiveness. For example does the question assume cases that may not be true?
2. Of the issues you identified, identify the most immediate, likely, and realistic issue, which may limit the effectiveness of this question in informing decision-makers of how they should address the central challenge of this survey.
3. Based on the issue you selected, please write a 3 questions that may serve as superior alternatives to the one listed above, and which overcome the issue you found to be most prevalent.
4. Output the reworded questions in the requested format.


In [85]:
# Create a module
class RewordQuestionModule(dspy.Module):
    def __init__(self, vary_temp=False):

        super().__init__()

        if vary_temp:
            rand_int = random.randint(1, 100)
        else:
            rand_int = 0
        
        self.rewordings = dspy.ChainOfThought(RewordQuestion, temperature=0.7+0.0001*rand_int)

    def forward(self, question, format):

        # this needs to return a dict and not a string for Evaluate to work
        return self.rewordings(question=question, format=format)

In [86]:
# Test out RewordQuestion

# Create a test input
test_input = {
    "response_format": "open",
    "description": "A question about park amenities",
    "main_text": "What kinds of amenities do you use most in local parks?",
    "response_categories": []
}

test_input_str = json.dumps(test_input)

format = "text survey"

# Create a RewordQuestionModule object
reword_questions = RewordQuestionModule()

# Run the test input
output = reword_questions(question = test_input_str, format = format)
rewordings = output.rewordings
print(f"Rationale is {output.rationale}")

# try to convert to json
try:
    rewordings = json.loads(rewordings)
    print("Correct output format")
    print(json.dumps(rewordings, indent=4))
except:
    print("Wrong output format")
    print(rewordings)

Rationale is produce the rewordings. We need to ensure that the question is clear, specific, and relevant to the central challenge of the survey.
Correct output format
[
    {
        "response_format": "open",
        "description": "A question about park amenities",
        "main_text": "What specific amenities would you like to see improved or added in local parks?",
        "response_categories": []
    },
    {
        "response_format": "open",
        "description": "A question about park amenities",
        "main_text": "How do park amenities impact your decision to visit local parks?",
        "response_categories": []
    },
    {
        "response_format": "open",
        "description": "A question about park amenities",
        "main_text": "In what ways do park amenities contribute to your overall park experience?",
        "response_categories": []
    }
]


In [87]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Please carefully review the inputted question, which will be asked of community members through a particular format that is also inputted.

Please think step-by-step and follow these instructions carefully:

1. Read the inputted question and identify 3 possible issues that may limit its effectiveness. For example does the question assume cases that may not be true?
2. Of the issues you identified, identify the most immediate, likely, and realistic issue, which may limit the effectiveness of this question in informing decision-makers of how they should address the central challenge of this survey.
3. Based on the issue you selected, please write a 3 questions that may serve as superior alternatives to the one listed above, and which overcome the issue you found to be most prevalent.
4. Output the reworded questions in the requested format.

---

Follow the following format.

Question: The question to reword. The input will be a JSON with the following structure: { "response_format":